# Diagnostik dataset v2 — di mana sinyal itu berada?
Hanya membaca `train.csv` dan `dev_test.csv` (TIDAK menyentuh lockbox). Tanpa GPU, beberapa menit.
Seluruh keluaran ditulis juga ke `../reports/diagnostik_v2.txt` agar tidak terpotong oleh tampilan notebook.

Pertanyaan: (1) bagian teks mana yang memuat sinyal kelas (judul, isi, awal, tengah, akhir)?
(2) adakah kata 'vonis' yang masih bocor? (3) apakah klasifikator sederhana menggeneralisasi ke SUMBER yang tidak dilihat?
(4) fitur permukaan mana di balik kegagalan C4?  (5) contoh acak untuk dibaca manusia.

In [1]:
import os, re, sys, io, contextlib
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

D = os.environ.get("V2_DIR") or ("../data/processed_v2" if os.path.exists("../data/processed_v2") else "data/processed_v2")
REP = os.environ.get("REP_DIR") or ("../reports" if os.path.exists("../reports") else ".")
os.makedirs(REP, exist_ok=True)
tr, te = pd.read_csv(os.path.join(D, "train.csv")), pd.read_csv(os.path.join(D, "dev_test.csv"))
assert "LOCKBOX" not in D.upper()
_log = io.StringIO()
def say(*a):
    s = " ".join(str(x) for x in a); print(s); _log.write(s + "\n")
say(f"train={len(tr):,} dev_test={len(te):,} | kolom: {list(tr.columns)}")

def seg(t, part, k=8):
    w = str(t).split()
    return " ".join(w[:k]) if part == "awal" else " ".join(w[-k:]) if part == "akhir" else " ".join(w[max(0, len(w)//2 - k//2):][:k])

def probe(xtr, xte, ytr, yte, ngram=(1, 2)):
    v = TfidfVectorizer(ngram_range=ngram, min_df=2, sublinear_tf=True)
    clf = LogisticRegression(max_iter=2000).fit(v.fit_transform(xtr), ytr)
    return accuracy_score(yte, clf.predict(v.transform(xte)))

train=4,073 dev_test=581 | kolom: ['id', 'text', 'title', 'body', 'label', 'sub_topic', 'source', 'group_id', 'date']


In [2]:
# Ganti blok views di Cell 2 dengan kode aman ini:
tr_body = tr.get("body", tr.text).fillna("").astype(str)
te_body = te.get("body", te.text).fillna("").astype(str)
tr_title = tr.get("title", tr.text.map(lambda t: seg(t, "awal", 12))).fillna("").astype(str)
te_title = te.get("title", te.text.map(lambda t: seg(t, "awal", 12))).fillna("").astype(str)

views = {
    "teks lengkap":            (tr.text.fillna("").astype(str), te.text.fillna("").astype(str)),
    "hanya JUDUL":             (tr_title, te_title),
    "hanya ISI (tanpa judul)": (tr_body, te_body),
    "8 kata awal":             (tr.text.fillna("").map(lambda t: seg(t, "awal")), te.text.fillna("").map(lambda t: seg(t, "awal"))),
    "8 kata tengah":           (tr.text.fillna("").map(lambda t: seg(t, "tengah")), te.text.fillna("").map(lambda t: seg(t, "tengah"))),
    "8 kata akhir":            (tr.text.fillna("").map(lambda t: seg(t, "akhir")), te.text.fillna("").map(lambda t: seg(t, "akhir"))),
    "isi, tanpa 8 kata awal":  (tr_body.map(lambda t: " ".join(str(t).split()[8:])), te_body.map(lambda t: " ".join(str(t).split()[8:]))),
}
rows = [(n, round(probe(a, b, tr.label, te.label) * 100, 2)) for n, (a, b) in views.items()]
say("\n[1] Akurasi probe linier per bagian teks:"); say(pd.DataFrame(rows, columns=["bagian", "akurasi_%"]).to_string(index=False))


[1] Akurasi probe linier per bagian teks:
                 bagian  akurasi_%
           teks lengkap      92.60
            hanya JUDUL      88.64
hanya ISI (tanpa judul)      93.46
            8 kata awal      86.23
          8 kata tengah      76.08
           8 kata akhir      79.17
 isi, tanpa 8 kata awal      88.81


In [3]:
# ---- 2. Kata 'vonis' yang mungkin masih bocor + token paling timpang ----
VERDICT = ["hoaks", "hoax", "disinformasi", "misinformasi", "klarifikasi", "cek fakta", "periksa fakta", "faktanya", "fakta sebenarnya",
           "tidak benar", "keliru", "bohong", "palsu", "penipuan", "hasil penelusuran", "turnbackhoax", "mafindo", "komdigi", "kominfo"]
low = tr.text.astype(str).str.lower()
vt = pd.DataFrame({k: low.str.contains(r"(?<!\w)" + re.escape(k) + r"(?!\w)").groupby(tr.label).mean() * 100 for k in VERDICT}).T.round(2)
vt.columns = ["fakta_%", "hoaks_%"]; vt["selisih"] = (vt["hoaks_%"] - vt["fakta_%"]).round(2)
say("\n[2a] Kata-vonis per label (train, % teks):"); say(vt.sort_values("selisih", ascending=False).head(10).to_string())

v = TfidfVectorizer(ngram_range=(1, 1), min_df=20, binary=True, use_idf=False, norm=None)
X = v.fit_transform(tr.text.map(lambda t: seg(t, "awal", 8)).astype(str)); y = tr.label.to_numpy()
p1 = (np.asarray(X[y == 1].sum(0)).ravel() + 1) / ((y == 1).sum() + 2); p0 = (np.asarray(X[y == 0].sum(0)).ravel() + 1) / ((y == 0).sum() + 2)
lo = pd.DataFrame({"token": v.get_feature_names_out(), "hoaks_%": (p1 * 100).round(1), "fakta_%": (p0 * 100).round(1), "log_odds": np.log(p1 / p0).round(2)})
say("\n[2b] Token paling timpang di 8 kata awal (min 20 kemunculan):")
say("  condong HOAKS:\n" + lo.sort_values("log_odds", ascending=False).head(12).to_string(index=False))
say("  condong FAKTA:\n" + lo.sort_values("log_odds").head(12).to_string(index=False))


[2a] Kata-vonis per label (train, % teks):
              fakta_%  hoaks_%  selisih
hoax             0.05     2.11     2.06
klarifikasi      0.39     1.67     1.28
palsu            0.00     1.13     1.13
tidak benar      0.05     0.79     0.74
bohong           0.00     0.34     0.34
faktanya         0.10     0.29     0.19
hoaks            0.34     0.49     0.15
kominfo          0.10     0.25     0.15
disinformasi     0.05     0.10     0.05
penipuan         0.10     0.15     0.05

[2b] Token paling timpang di 8 kata awal (min 20 kemunculan):
  condong HOAKS:
      token  hoaks_%  fakta_%  log_odds
   whatsapp      3.6      0.0      4.30
    artikel      3.0      0.0      4.13
     gambar      2.3      0.0      3.83
  tangkapan      2.2      0.0      3.81
   berjudul      1.4      0.0      3.37
      video      9.5      0.3      3.32
   ternyata      1.3      0.0      3.26
     corona      1.1      0.0      3.14
      layar      2.3      0.1      3.14
   lowongan      1.1      0.0      3

In [4]:
# ---- 3. Generalisasi ke SUMBER yang tidak dilihat (leave-one-source-out, probe linier) ----
say("\n[3] Leave-one-source-out: latih tanpa sumber S, ukur recall kelas S pada teks S (dev_test + train sumber itu):")
full = pd.concat([tr, te], ignore_index=True); out = []
for s in sorted(full.source.unique()):
    held = full[full.source == s]; rest = full[full.source != s]
    v2 = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    clf = LogisticRegression(max_iter=2000).fit(v2.fit_transform(rest.text.astype(str)), rest.label)
    rec = (clf.predict(v2.transform(held.text.astype(str))) == held.label.iloc[0]).mean()
    out.append((s, int(held.label.iloc[0]), len(held), round(rec * 100, 2)))
say(pd.DataFrame(out, columns=["sumber_ditahan", "label", "n", "recall_%"]).to_string(index=False))
say("    Catatan: kelas sumber yang ditahan masih punya sumber lain di data latih; ini probe, bukan bukti generalisasi sejati.")


[3] Leave-one-source-out: latih tanpa sumber S, ukur recall kelas S pada teks S (dev_test + train sumber itu):
  sumber_ditahan  label    n  recall_%
cnnindonesia.com      0  601     85.52
hoax_archive_15k      1  651     75.58
   komdigi.go.id      1   42     78.57
      kompas.com      0  381     73.75
        tempo.co      0 1345     22.30
 turnbackhoax.id      1 1634     32.99
    Catatan: kelas sumber yang ditahan masih punya sumber lain di data latih; ini probe, bukan bukti generalisasi sejati.


In [5]:
# ---- 4. Fitur permukaan di balik C4 ----
def feats(d):
    t = d.text.astype(str); w = t.str.split().str.len()
    return pd.DataFrame({"n_kata": w, "n_karakter": t.str.len(), "karakter_per_kata": (t.str.len() / w).round(3),
                         "n_koma": t.str.count(","), "n_titik_tanya_seru": t.str.count(r"[.!?]"), "n_kutip": t.str.count(r'["“”]'),
                         "n_angka": t.str.count(r"\d"), "pct_kapital": t.apply(lambda s: sum(c.isupper() for c in s) / max(len(s), 1) * 100).round(3)})
Ftr, Fte = feats(tr), feats(te); res = []
for c in Ftr.columns:
    a = accuracy_score(te.label, DecisionTreeClassifier(max_depth=3, random_state=0).fit(Ftr[[c]], tr.label).predict(Fte[[c]]))
    res.append((c, round(a * 100, 1), round(Ftr[c][tr.label == 0].mean(), 3), round(Ftr[c][tr.label == 1].mean(), 3)))
say("\n[4] Akurasi pohon per SATU fitur permukaan (dev_test) dan rata-rata per label (train):")
say(pd.DataFrame(res, columns=["fitur", "akurasi_%", "rata2_fakta", "rata2_hoaks"]).sort_values("akurasi_%", ascending=False).to_string(index=False))
allf = accuracy_score(te.label, DecisionTreeClassifier(max_depth=4, random_state=0).fit(Ftr, tr.label).predict(Fte))
say(f"    Semua fitur bersama (pohon kedalaman 4): {allf*100:.1f}%")


[4] Akurasi pohon per SATU fitur permukaan (dev_test) dan rata-rata per label (train):
             fitur  akurasi_%  rata2_fakta  rata2_hoaks
n_titik_tanya_seru       65.7        3.418        4.188
            n_koma       62.8        2.186        1.461
 karakter_per_kata       61.4        7.158        6.951
       pct_kapital       60.8       10.083       18.382
           n_kutip       59.9        0.537        0.678
            n_kata       55.4       39.694       38.763
        n_karakter       53.0      285.880      269.982
           n_angka       51.8        3.615        4.446
    Semua fitur bersama (pohon kedalaman 4): 75.0%


In [6]:
# ---- 5. Baca dengan mata: 6 contoh acak per label ----
say("\n[5] Contoh acak (baca dan nilai wajar-tidaknya pembentukan teks):")
for lab in (0, 1):
    say(f"--- label {lab} ---")
    for _, r in tr[tr.label == lab].sample(6, random_state=7).iterrows():
        say(f"  [{r.source}] {str(r.text)[:200]}")
open(os.path.join(REP, "diagnostik_v2.txt"), "w", encoding="utf-8").write(_log.getvalue())
print("\nTersimpan penuh di:", os.path.abspath(os.path.join(REP, "diagnostik_v2.txt")))


[5] Contoh acak (baca dan nilai wajar-tidaknya pembentukan teks):
--- label 0 ---
  [tempo.co] Jokowi Ajak Masyarakat Hadapi 2022 dengan Semangat Baru. commerce, misalnya, mencapai nilai 24,8 miliar dolar AS tahun ini," ujarnya di akun media sosial instagram @jokowi.
  [kompas.com] Jokowi dan Purnawirawan TNI Bahas Kondisi Politik dan Keamanan. Presiden Joko Widodo dan para purnawirawan TNI membahas kondisi bangsa terkini saat bertemu di Istana Merdeka Jakarta, Jumat (31/5/2019)
  [tempo.co] Haedar Nashir Minta Menteri Tak Manfaatkan Jabatan untuk Pemilu 2024. masing," ucap dia.
  [tempo.co] Kapolri Mutasi 215 Pati Polisi: Penggantian Wakalemdiklat hingga Kapolda Sulbar. Kapolri Jenderal Listyo Sigit Prabowo memutasi dan merotasi 215 perwira polisi dalam rangka naik jabatan dan pensiun, s
  [cnnindonesia.com] Ketua Komisi X Sebut Penyebaran Radikalisme di Kampus Nyata. berbasis teman sebaya. Menurutnya, setiap mahasiswa bisa saling mengawasi dan mengingatkan jika melihat perubahan per